<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 内存高效的模型权重加载

- 本 notebook 提供在 GPU（或 CPU）内存有限时加载较大预训练或微调模型的实用技巧
- 具体而言，它针对你使用 `torch.save(model.state_dict(), "model.pth")` 保存模型（例如在第 5–7 章中）并希望在新的会话中稍后加载以继续预训练或进一步微调的情况
- 虽然示例使用的是 LLM，但本 notebook 中讲解的方法具有通用性，适用于加载任何 PyTorch 模型，而不仅仅是 LLM

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/memory-efficient-loading/memory-efficient-loading.webp" width="800px">

In [1]:
from importlib.metadata import version

pkgs = [
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

torch version: 2.9.1+cu130


&nbsp;
## 1. 基准测试工具

- 首先，我们定义一些用于跟踪 VRAM（GPU 内存）的工具代码
- 稍后，我们还会引入一个用于跟踪主系统 RAM（CPU 内存）的工具
- 这些函数的用途将在后面实际应用时变得清晰

In [2]:
import gc
import time
import torch


def start_memory_tracking():
    """初始化 GPU 内存跟踪。"""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    else:
        print("本 notebook 面向 CUDA GPU，但当前不可用 CUDA。")

def print_memory_usage():
    max_gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)  # 将字节转换为 GB
    print(f"已分配 GPU 内存峰值：{max_gpu_memory:.1f} GB")

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)  # 留出缓冲时间以便内存释放
    torch.cuda.reset_peak_memory_stats()
    max_memory_allocated = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    print(f"已分配 GPU 内存峰值：{max_memory_allocated:.1f} GB")

&nbsp;
## 2. 模型设置

- 本节代码用于设置模型本身
- 这里我们使用 "large" 规模的 GPT-2 模型以使示例更有趣（你也可以使用 "gpt2-small (124M)" 来降低本 notebook 的内存需求和运行时间）

In [3]:
from previous_chapters import GPTModel
# 如果本地没有 `previous_chapters.py` 文件，
# 可以从 `llms-from-scratch` PyPI 包中导入。
# 详情见：https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 例如：
# from llms_from_scratch.ch04 import GPTModel



BASE_CONFIG = {
    "vocab_size": 50257,     # 词表大小
    "context_length": 1024,  # 上下文长度
    "drop_rate": 0.0,        # Dropout 比率
    "qkv_bias": True         # Query-Key-Value 偏置
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-xl (1558M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

- 现在，让我们看看 GPU 内存跟踪函数的实际效果：

In [4]:
start_memory_tracking()


model = GPTModel(BASE_CONFIG)
device = torch.device("cuda")
model.to(device)

print_memory_usage()

/home/rasbt/jupyterlab/reasoning/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


已分配 GPU 内存峰值：6.4 GB


- 此外，让我们传入示例张量，确保模型运行正常

In [5]:
# 测试模型是否正常工作（此处无需跟踪内存）
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

- 接下来，假设我们正在预训练模型并将其保存以供后续使用
- 为简化起见，此处跳过实际预训练，仅保存初始化后的模型（但概念相同）

In [6]:
# 训练代码应放在此处...

model.train()
torch.save(model.state_dict(), "model.pth")

- 最后，我们在 Python 会话中删除模型和示例张量，以重置 GPU 内存

In [7]:
del model, test_input
cleanup()

已分配 GPU 内存峰值：0.0 GB


&nbsp;
## 3. 基本权重加载

- 现在开始有趣的部分：加载预训练模型权重
- 让我们看看加载先前保存的模型需要多少 GPU 内存

In [8]:
# 然后加载预训练权重

start_memory_tracking()

model = GPTModel(BASE_CONFIG)
model.to(device)

model.load_state_dict(
    torch.load("model.pth", map_location=device, weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

已分配 GPU 内存峰值：12.8 GB


- 请注意，内存占用是上一会话中的 2 倍
- 这是因为在短时间内，同一份模型权重会同时在内存中存在两份：
  - 第一次通过 `model.to(device)`
  - 第二次通过代码行 `model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))`；最终，加载的模型权重会被复制到模型中，`state_dict` 会被丢弃，但在短暂的时间内，主模型和加载的 `state_dict` 会同时存在于内存中
- 其余章节将重点解决这一问题
- 但首先，让我们测试模型并重置 GPU 内存


In [9]:
# 测试模型是否正常工作（此处无需跟踪内存）
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

已分配 GPU 内存峰值：0.0 GB


- 让我们测试另一种在实践中非常常见的模式：

In [10]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG)

model.load_state_dict(
    torch.load("model.pth", map_location="cpu", weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

已分配 GPU 内存峰值：6.4 GB


In [11]:
# 测试模型是否正常工作（此处无需跟踪内存）
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

已分配 GPU 内存峰值：0.0 GB


- 因此，就峰值内存而言，是先在设备上实例化模型再使用 `map_location="device"`，还是先将权重加载到 CPU 内存（`map_location="cpu"`）再移动到设备，并没有区别

&nbsp;
## 4. 顺序加载权重

- 针对上一节强调的模型权重在 GPU 内存中重复占用的问题，一种变通方法是顺序加载模型
- 下面我们将：
  - 首先将模型加载到 GPU 内存
  - 然后将模型权重加载到 CPU 内存
  - 最后逐个将参数复制到 GPU 内存


In [ ]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG).to(device)

state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

print_memory_usage()

# 顺序将权重复制到模型参数
with torch.no_grad():
    for name, param in model.named_parameters():
        if name in state_dict:
            param.copy_(state_dict[name].to(device))
        else:
            print(f"警告：在 state_dict 中未找到 {name}。")

print_memory_usage()

已分配 GPU 内存峰值：6.4 GB
已分配 GPU 内存峰值：6.7 GB


- 如上所示，内存占用比之前低得多
- 请注意，内存从 6.4 GB 增加到 6.7 GB，因为最初内存中只有模型，随后内存中有模型加上 1 个参数张量（我们临时将参数张量移动到 GPU，以便使用 `".to"` 将其赋值给模型）
- 总体而言，这是一个显著的改进
- 再次简要测试模型，然后为下一节重置 GPU 内存

In [ ]:
# 测试模型是否正常工作（此处无需跟踪内存）
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input, state_dict, param
cleanup()

已分配 GPU 内存峰值：0.0 GB


&nbsp;
## 5. 以低 CPU 内存方式加载模型

- 在上一节中，我们通过先将权重（`state_dict`）加载到 CPU 内存，再逐个复制到模型中来降低 GPU 内存占用
- 然而，如果 CPU 内存有限，我们该怎么办？
- 本节使用 PyTorch 所谓的 `"meta"` 设备方法，在 GPU 内存较大但 CPU 内存较小的机器上加载模型
- 但首先，让我们定义一个便于监控 CPU 内存的辅助函数

In [ ]:
import os
import psutil
from threading import Thread


def memory_usage_in_gb(func, *args, **kwargs):
    process = psutil.Process(os.getpid())

    # 在运行函数前测量基线内存占用
    baseline_mem = process.memory_info().rss / 1024 ** 3  # 单位：GB

    # 在单独线程中开始监控内存
    mem_usage = []
    done = False

    def monitor_memory():
        while not done:
            mem_usage.append(process.memory_info().rss / 1024 ** 3)  # 转换为 GB
            time.sleep(0.1)

    t = Thread(target=monitor_memory)
    t.start()

    # 运行函数
    func(*args, **kwargs)

    # 停止监控
    done = True
    t.join()

    peak_mem_usage_gb = max(mem_usage) - baseline_mem
    return peak_mem_usage_gb


- 首先，让我们跟踪上一节顺序加载权重方法的 CPU 内存占用

In [ ]:
def load_sequentially():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG).to(device)

    state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

    print_memory_usage()

    # 顺序将权重复制到模型参数
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))
            else:
                print(f"警告：在 state_dict 中未找到 {name}。")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_sequentially)
print(f"-> 已分配 CPU 内存峰值：{peak_memory_used:.1f} GB")

已分配 GPU 内存峰值：6.4 GB
已分配 GPU 内存峰值：6.7 GB
-> 已分配 CPU 内存峰值：6.3 GB


- 现在，假设我们有一台 CPU 内存较小但 GPU 内存较大的机器
- 我们可以通过引入 PyTorch 所谓的 "meta" 设备来权衡 CPU 内存与 GPU 内存的占用
- PyTorch 的 meta 设备是一种特殊设备类型，允许你创建张量而无需为数据分配实际内存，从而有效创建 "meta" 张量
- 这对于模型分析或架构定义等任务很有用，在这些任务中你需要张量的形状和类型，但不需要内存分配的开销

In [ ]:
def load_sequentially_with_meta():
    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    state_dict = torch.load("model.pth", map_location=device, weights_only=True)

    print_memory_usage()

    # 顺序将权重复制到模型参数
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name])
            else:
                print(f"警告：在 state_dict 中未找到 {name}。")

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(load_sequentially_with_meta)
print(f"-> 已分配 CPU 内存峰值：{peak_memory_used:.1f} GB")

已分配 GPU 内存峰值：12.8 GB
已分配 GPU 内存峰值：12.8 GB
-> 已分配 CPU 内存峰值：1.3 GB


- 如上所示，通过在 meta 设备上创建模型并将权重直接加载到 GPU 内存，我们有效降低了 CPU 内存需求
- 有人可能会问："那么顺序加载权重是否仍然必要？它与原始方法相比如何？"
- 让我们检查一下简单的 PyTorch 权重加载方法以作对比（来自本 notebook 第一节的权重加载部分）：

In [ ]:
def baseline():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG)
    model.to(device)

    model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
    model.to(device)
    model.eval();

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(baseline)
print(f"-> 已分配 CPU 内存峰值：{peak_memory_used:.1f} GB")

已分配 GPU 内存峰值：12.8 GB
-> 已分配 CPU 内存峰值：4.4 GB


- 如上所示，不使用 meta 设备的"简单"权重加载会占用更多内存
- 换句话说，如果你的机器 CPU 内存有限，可以使用 meta 设备方法将模型权重直接加载到 GPU 内存，以降低 CPU 内存峰值占用

&nbsp;
## 6. 使用 `mmap=True`（推荐）

- 作为中级或高级 `torch.load` 用户，你可能会好奇这些方法与 PyTorch 中 `mmap=True` 设置的对比
- PyTorch 中的 `mmap=True` 设置启用内存映射文件 I/O，允许张量直接从磁盘存储访问数据，从而在 RAM 有限时不将整个文件加载到 RAM 中，从而降低内存占用
- 另请参阅 [mikaylagawarecki](https://github.com/rasbt/LLMs-from-scratch/issues/402) 的有益评论
- 乍一看，它可能不如上面的顺序加载方法高效：

In [ ]:
def best_practices():
  with torch.device("meta"):
      model = GPTModel(BASE_CONFIG)

  model.load_state_dict(
      torch.load("model.pth", map_location=device, weights_only=True, mmap=True),
      assign=True
  )

  print_memory_usage()

peak_memory_used = memory_usage_in_gb(best_practices)
print(f"-> 已分配 CPU 内存峰值：{peak_memory_used:.1f} GB")

已分配 GPU 内存峰值：6.4 GB
-> 已分配 CPU 内存峰值：5.9 GB


- CPU RAM 占用如此之高的原因是这台机器上有足够的 CPU RAM 可用
- 然而，如果你在 CPU RAM 有限的机器上运行，`mmap` 方法会使用更少的内存

&nbsp;
## 7. 其他方法

- 本 notebook 聚焦于 PyTorch 中用于加载权重的简单内置方法
- 对于 CPU 内存有限的情况，推荐使用上文介绍的 `mmap=True` 方法
- 另外，还有一种暴力方法是分别保存和加载每个权重张量：

In [ ]:
model = GPTModel(BASE_CONFIG)
# 假设 `model` 是你训练好的模型
state_dict = model.state_dict()

# 创建目录以存储各个参数文件
os.makedirs("model_parameters", exist_ok=True)

# 分别保存每个参数张量
for name, param in state_dict.items():
    torch.save(param.cpu(), f"model_parameters/{name}.pt")

del model

In [ ]:
def load_individual_weights():

    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    print_memory_usage()
    param_dir = "model_parameters"

    with torch.no_grad():
        for name, param in model.named_parameters():
            weight_path = os.path.join(param_dir, f"{name}.pt")
            if os.path.exists(weight_path):
                param_data = torch.load(weight_path, map_location="cpu", weights_only=True)
                param.copy_(param_data)
                del param_data  # 释放内存
            else:
                print(f"警告：在 {param_dir} 中未找到 {name}。")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_individual_weights)
print(f"-> 已分配 CPU 内存峰值：{peak_memory_used:.1f} GB")

已分配 GPU 内存峰值：6.4 GB
已分配 GPU 内存峰值：6.4 GB
-> 已分配 CPU 内存峰值：0.3 GB
